<div class="blog-language-switch" role="group" aria-label="文章语言">
<a href="../../Deep-Learning/06-generalization-regularization-experiments.html" lang="en" hreflang="en">English</a>
<span aria-current="page">中文</span>
</div>

[返回深度学习总览](Deep-Learning.html)

## **泛化、正则化与实验实践** {#generalization-regularization-experimental-practice}

第 05 章解释了损失、梯度和优化器如何降低已观测样本上的误差。本章讨论更困难的问题：**这种下降揭示的是可以复用的规律，还是只是对训练集更好的记忆？** 泛化（generalization）是学习到的预测器在来自部署总体的新样本上仍然有效的能力。正则化改变哪些解可达或更受偏好；实验实践则决定表面上的改进是否可信。

对于从未知分布 $\mathcal{P}$ 抽取的训练样本 $S=\{(x_i,y_i)\}_{i=1}^{N}$，经验风险与总体风险分别为

$$
\widehat R_S(\theta)=\frac{1}{N}\sum_{i=1}^{N}\ell(f_\theta(x_i),y_i),
\qquad
R(\theta)=\mathbb{E}_{(x,y)\sim\mathcal{P}}[\ell(f_\theta(x),y)].
$$

训练能够直接优化 $\widehat R_S$，部署真正关心的是 $R$。验证集和测试集只是总体行为的有限样本估计，而不是可以互换的额外训练数据。验证集用于架构、正则化强度和停止时刻等决策；测试集则应在这些决策冻结后才用于最终估计。

### **过参数化网络中的泛化** {#generalization-overparameterized-networks}

当模型拥有足够自由度来插值训练集时，它就是**过参数化（overparameterized）**的，参数数量甚至常常多于训练样本。经典容量观点提醒我们，更大的假设空间能够拟合更多偶然模式。现代神经网络让这一认识变得更细致：两个网络都可以达到零训练误差，却在观测样本之间表示完全不同的函数。架构、初始化、随机优化、归一化、数据增强、间隔和参数范数共同从许多插值解中选择一个解。

一种通用的泛化陈述具有如下形式：

$$
R(\theta) \lesssim \widehat R_S(\theta)
+\mathcal{C}(f_\theta,S)
+O\!\left(\sqrt{\frac{\log(1/\delta)}{N}}\right),
$$

该关系以至少 $1-\delta$ 的概率成立。这里的 $\mathcal{C}$ 是一种通常依赖数据的**有效复杂度**。不同理论会通过间隔、范数、稳定性、压缩长度、PAC-Bayes 量或其他受限描述来定义它。这个式子是概念模板，而不是可以对所有网络直接代数代入的统一定理。仅靠原始参数数量无法解释某个训练完成的网络为什么能够迁移。

本章所有可运行示例都沿用一条实验主线：scikit-learn 从 UCI 仓库分发的 1,797 样本 **Optical Recognition of Handwritten Digits** 数据集。于是每个示例都是同一个十分类问题上的受控改动，而不是互不相关的玩具实验。每张图像为 $8\times8$ 灰度数组，像素值在 0 到 16 之间。我们把像素缩放到 $[0,1]$，固定一个分层的 60/20/20 划分，并且绝不使用测试标签选择模型。选择这个小型数据集是有意的：所有实验都能在 CPU 上运行，同时保留真实的抽样、多分类和图像增强问题。

数据来源：[scikit-learn `load_digits`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html) 与 [UCI Optical Recognition of Handwritten Digits](https://archive.ics.uci.edu/dataset/80/optical%2Brecognition%2Bof%2B)（CC BY 4.0，DOI `10.24432/C50P49`）。

<details>
<summary><strong>PyTorch：建立共享的 Digits 实验</strong></summary>

```python
import copy
import math
import random
import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)


def seed_everything(seed=606):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()
digits = load_digits()
images = torch.tensor(digits.images, dtype=torch.float32).unsqueeze(1) / 16.0
targets = torch.tensor(digits.target, dtype=torch.long)
all_indices = np.arange(len(targets))

# The test set is separated first; validation is then separated from development data.
dev_idx, test_idx = train_test_split(
    all_indices, test_size=0.20, random_state=606, stratify=digits.target
)
train_idx, val_idx = train_test_split(
    dev_idx, test_size=0.25, random_state=606, stratify=digits.target[dev_idx]
)

x_train, y_train = images[train_idx], targets[train_idx]
x_val, y_val = images[val_idx], targets[val_idx]
x_test, y_test = images[test_idx], targets[test_idx]


def make_loader(x, y, batch_size=128, shuffle=False, seed=606):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        TensorDataset(x, y), batch_size=batch_size, shuffle=shuffle, generator=generator
    )


class DigitsMLP(nn.Module):
    def __init__(self, widths=(128, 64), dropout=0.0):
        super().__init__()
        layers, in_features = [], 64
        for width in widths:
            layers += [nn.Linear(in_features, width), nn.ReLU(), nn.Dropout(dropout)]
            in_features = width
        layers.append(nn.Linear(in_features, 10))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x.flatten(1))


@torch.no_grad()
def classification_metrics(model, x, y):
    model.eval()
    logits = model(x)
    return {
        "loss": F.cross_entropy(logits, y).item(),
        "accuracy": (logits.argmax(1) == y).float().mean().item(),
    }


def fit_model(model, epochs=30, lr=3e-3, weight_decay=0.0, seed=606,
              train_x=x_train, train_y=y_train, callback=None):
    seed_everything(seed)
    loader = make_loader(train_x, train_y, shuffle=True, seed=seed)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = []
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(xb), yb)
            loss.backward()
            optimizer.step()
        record = {
            "epoch": epoch,
            "train": classification_metrics(model, train_x, train_y),
            "validation": classification_metrics(model, x_val, y_val),
        }
        history.append(record)
        if callback is not None and callback(model, record):
            break
    return history


assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert x_train.shape[1:] == (1, 8, 8)
assert y_train.unique().numel() == 10
print({"train": len(train_idx), "validation": len(val_idx), "test": len(test_idx)})
```

</details>

上面的数据划分、预处理和评估函数构成了本章的**实验契约**。后续章节可以改变某个处理因素，但会复用相同的数据边界和指标定义。这样比较才具有解释性，意外的数据泄漏也更容易被检测。

### **欠拟合、过拟合与泛化间隙** {#underfitting-overfitting-generalization-gap}

当整个学习流程无法表示或优化出有效规律时，就会出现**欠拟合（underfitting）**：训练和验证表现都很差。**过拟合（overfitting）**则指继续拟合能够改善训练集，但验证表现停滞或恶化。这些诊断属于完整流程，不是永久附着在某种架构上的标签。一个小模型可能在某种特征表示下欠拟合，却能在改进预处理后表现良好；一个大模型在强数据增强下可以泛化，却可能在无增强时记住受污染的标签。

在指标与评估流程一致时，可以把验证损失间隙写成

$$
G_{\mathrm{val}}=\widehat R_{\mathrm{val}}(\theta)-\widehat R_{\mathrm{train}}(\theta).
$$

正的 $G_{\mathrm{val}}$ 只是行为差异的证据，并不是完整的因果诊断。训练期间，增强和 Dropout 会有意让训练样本更难；评估时它们被关闭。因此即使系统没有问题，训练损失也可能高于验证损失。在解释间隙前，必须比较模型模式、预处理、归约方式和类别组成。

学习曲线可以区分三种常见情况：训练与验证损失都高通常意味着表示或优化失败；训练损失低而验证损失明显更高，暗示过度拟合或分布不匹配；两者同步下降则说明继续优化仍有价值。还应同时报告任务指标，因为交叉熵的小幅变化可能主要反映置信度校准，而不是预测类别发生变化。

<details>
<summary><strong>PyTorch：在共享划分上诊断容量与训练时间</strong></summary>

```python
seed_everything(607)
linear_model = DigitsMLP(widths=())
linear_history = fit_model(linear_model, epochs=20, lr=5e-3, seed=607)

seed_everything(607)
wide_model = DigitsMLP(widths=(256, 256))
wide_history = fit_model(wide_model, epochs=70, lr=3e-3, seed=607)

checkpoints = [0, 4, 14, 29, 69]
wide_curve = [
    (
        wide_history[i]["epoch"] + 1,
        round(wide_history[i]["train"]["loss"], 4),
        round(wide_history[i]["validation"]["loss"], 4),
    )
    for i in checkpoints
]

assert wide_history[-1]["train"]["loss"] < linear_history[-1]["train"]["loss"]
assert all(math.isfinite(value) for row in wide_curve for value in row[1:])
print("linear:", linear_history[-1])
print("wide learning curve (epoch, train loss, validation loss):", wide_curve)
```

</details>

![多项式模型展示了随着容量增加出现的欠拟合、适当拟合与过拟合。](assets/dl06-underfitting-overfitting.png){fig-align="center" width="76%" fig-alt="三幅多项式回归图，对比模型容量增加时的欠拟合、适当拟合和过拟合。"}

*图片来源：[scikit-learn, Underfitting vs. Overfitting](https://scikit-learn.org/stable/auto_examples/model_selection/plot_underfitting_overfitting.html)，BSD-3-Clause 文档示例。*

图中采用多项式回归，是因为拟合函数的形状可以直接观察。Digits 实验在分类任务中检验同一原则：应检查训练轨迹，而不能只根据参数数量给模型贴上“过拟合”标签。

### **显式与隐式正则化** {#explicit-implicit-regularization}

正则化是任何为了偏好预期能迁移的行为，而对学习过程施加约束、偏置或拟合牺牲的机制。它远不只是向损失加入一个惩罚项。

**显式正则化**直接出现在目标函数或训练变换中，例如范数惩罚、标签平滑、MixUp、数据增强和参数约束。若

$$
J(\theta)=\widehat R_S(\theta)+\lambda\Omega(\theta),
$$

$\Omega$ 衡量不希望出现的性质，$\lambda$ 控制其强度。更大的 $\lambda$ 并不普遍意味着更好的泛化；它会让估计器沿偏差-方差权衡移动，过强时最终导致欠拟合。

**隐式正则化**来自参数化方式与优化路径，即使书面目标函数中没有惩罚项也会存在。小批量噪声、初始化尺度、卷积权重共享、优化器选择、有限训练时间和早停都会使某些解比其他解更容易达到。例如，在从零初始化的欠定线性系统上，梯度下降会收敛到最小欧氏范数插值解。深层非线性网络不存在同样统一的规律，但原则仍然成立：优化器是解的选择机制，而不只是损失最小化器。

这个区分对实验很重要。如果同时改变权重衰减和数据增强，就无法判断改进来自范数控制还是不变性假设。严谨研究先一次改变一个因素，记录训练/验证行为，再通过后续因子实验报告交互作用。

<details>
<summary><strong>PyTorch：在显式约束下比较两个插值解</strong></summary>

```python
# Use a deterministic subset of real Digits pixels to create an underdetermined
# linear regression problem: 40 examples constrain 64 pixel coefficients.
design = x_train[:40].flatten(1)
binary_target = (y_train[:40] >= 5).float().unsqueeze(1)

minimum_norm = torch.linalg.pinv(design) @ binary_target
ridge_strength = 0.1
ridge = torch.linalg.solve(
    design.T @ design + ridge_strength * torch.eye(design.shape[1]),
    design.T @ binary_target,
)

minimum_norm_fit = F.mse_loss(design @ minimum_norm, binary_target)
ridge_fit = F.mse_loss(design @ ridge, binary_target)

# Ridge accepts more training error to shrink the coefficient vector.
assert ridge.norm() < minimum_norm.norm()
assert ridge_fit >= minimum_norm_fit - 1e-7
print({
    "minimum_norm": (minimum_norm_fit.item(), minimum_norm.norm().item()),
    "ridge": (ridge_fit.item(), ridge.norm().item()),
})
```

</details>

这个小计算隔离了从多个已拟合系数向量中选择解的问题。神经网络正则化更复杂，但诊断问题相同：**被选中解的哪种性质发生了变化，为什么这种性质应当有助于迁移？**

### **权重衰减与范数控制** {#weight-decay-norm-control}

$L_2$ 惩罚向损失加入 $\frac{\lambda}{2}\lVert\theta\rVert_2^2$，其梯度贡献为 $\lambda\theta$，因此普通 SGD 的更新是

$$
\theta_{t+1}
=\theta_t-\eta_t\left(g_t+\lambda\theta_t\right)
=(1-\eta_t\lambda)\theta_t-\eta_tg_t.
$$

对 SGD 而言，这在代数上等价于乘法式权重衰减。对于自适应优化器，把 $\lambda\theta$ 放入梯度还会让它经过逐坐标动量归一化。AdamW 把收缩与自适应梯度更新解耦：

$$
\theta_{t+1}=(1-\eta_t\lambda)\theta_t
-\eta_t\frac{\widehat m_t}{\sqrt{\widehat v_t}+\epsilon}.
$$

解耦让预期的收缩更容易解释，但有效衰减仍取决于学习率调度和更新次数。偏置以及归一化层的缩放/平移参数通常会被排除，因为收缩它们并不表达与收缩权重矩阵相同的函数偏好。因此参数分组属于方法本身。

范数控制不能保证函数一定简单。ReLU 网络允许相邻层之间进行补偿性重缩放，小参数范数也可能对应尖锐或校准不佳的决策边界。应记录验证指标和相关范数，而不能把范数当作部署目标直接优化。

<details>
<summary><strong>PyTorch：测量 Digits 上 AdamW 的收缩效果</strong></summary>

```python
decay_results = {}
for decay in (0.0, 1e-3, 1e-2):
    seed_everything(608)
    model = DigitsMLP(widths=(128, 64))
    history = fit_model(model, epochs=30, lr=3e-3, weight_decay=decay, seed=608)
    matrix_norm = torch.sqrt(sum(
        parameter.detach().pow(2).sum()
        for name, parameter in model.named_parameters()
        if parameter.ndim > 1
    )).item()
    decay_results[decay] = {
        "weight_norm": matrix_norm,
        "validation": history[-1]["validation"],
    }

assert decay_results[1e-2]["weight_norm"] < decay_results[0.0]["weight_norm"]
assert all(0.0 <= result["validation"]["accuracy"] <= 1.0 for result in decay_results.values())
print(decay_results)
```

</details>

在固定流程下，最强衰减应降低测得的矩阵范数，但验证准确率不必单调改善。科学上有意义的输出是多个合理强度组成的曲线，而不是“正则化越多越好”的结论。

### **Dropout 与随机深度** {#dropout-stochastic-depth}

Dropout 在训练期间随机移除激活。对于激活向量 $h$、独立掩码 $m_j\sim\mathrm{Bernoulli}(q)$ 和保留概率 $q=1-p$，**inverted dropout** 定义为

$$
\widetilde h=\frac{m\odot h}{q},
\qquad
\mathbb{E}[\widetilde h]=h.
$$

$1/q$ 因子保持激活期望不变，使评估时可以直接使用无掩码网络。Dropout 阻止单元过度依赖某种精确的协同适应模式，但效果依赖放置位置。对小模型使用高 Dropout，或在严重信息瓶颈后使用它，都可能破坏信号。模型还必须正确切换 `train()` 与 `eval()`，否则评估会持续随机，并可能出现系统性的尺度错误。

**随机深度（stochastic depth）**把同一思想应用于残差分支。对 $h_{l+1}=h_l+F_l(h_l)$，训练时可使用

$$
h_{l+1}=h_l+\frac{b_l}{q_l}F_l(h_l),
\qquad b_l\sim\mathrm{Bernoulli}(q_l).
$$

即使残差分支被丢弃，恒等路径依然存在，因此很深的网络相当于在一组不同有效深度上训练。存活概率常随深度降低，因为后部模块通常更冗余。该方法在结构上适合残差网络，并不是任意隐藏单元 Dropout 的直接替代品。

<details>
<summary><strong>PyTorch：在真实 digit 激活上验证随机机制</strong></summary>

```python
class ResidualMLPBlock(nn.Module):
    def __init__(self, width=64, drop_path=0.0):
        super().__init__()
        self.branch = nn.Sequential(nn.Linear(width, width), nn.ReLU(), nn.Linear(width, width))
        self.drop_path = drop_path

    def forward(self, x):
        residual = self.branch(x)
        if self.training and self.drop_path > 0:
            keep = 1.0 - self.drop_path
            mask = torch.bernoulli(torch.full((x.shape[0], 1), keep, device=x.device))
            residual = residual * mask / keep
        return x + residual


digit_vectors = x_train[:32].flatten(1)
dropout = nn.Dropout(p=0.5)
dropout.train()
samples = torch.stack([dropout(digit_vectors) for _ in range(200)])
empirical_mean = samples.mean(0)

block = ResidualMLPBlock(drop_path=0.5)
block.train()
block_outputs = torch.stack([block(digit_vectors) for _ in range(100)])
block.eval()
deterministic_a = block(digit_vectors)
deterministic_b = block(digit_vectors)

assert (empirical_mean - digit_vectors).abs().mean() < 0.05
assert torch.allclose(deterministic_a, deterministic_b)
assert block_outputs.var(0).mean() > 0
print("dropout expectation error:", (empirical_mean - digit_vectors).abs().mean().item())
```

</details>

Dropout 改变激活层面的协同适应，随机深度改变路径层面的计算。比较时应匹配训练预算，并报告评估模式下的结果。训练模式中的方差是预期行为，不代表最终确定性网络本身存在不确定性。

### **数据增强** {#data-augmentation}

数据增强通过变换 $T\sim\mathcal{A}$ 编码任务假设，训练目标为

$$
\frac{1}{N}\sum_{i=1}^{N}
\mathbb{E}_{T\sim\mathcal{A}}
\left[\ell(f_\theta(T(x_i)),y_i)\right].
$$

它鼓励模型对保持标签不变的变换具有不变性。困难之处正是“**保持标签不变**”。小幅平移可能不会改变手写数字，大角度旋转却可能把清晰符号变得含糊。在医学图像中，左右翻转可能改变解剖含义；在遥感图像中则可能无害。因此增强是对允许变化的建模，而不是免费的数据。

应在输入空间检查增强强度，并按切片评估。如果变换样本看起来不真实，即使总体验证指标改善，也可能掩盖临床或业务关键子群的退化。Digits 只有 $8\times8$ 分辨率，插值尤其容易破坏结构，所以本章只采用小幅仿射平移，并明确限制结果范围。

<details>
<summary><strong>PyTorch：对 Digits 应用并验证小幅平移</strong></summary>

```python
def translate_digits(batch, max_pixels=1, seed=610):
    generator = torch.Generator().manual_seed(seed)
    n = batch.shape[0]
    shifts = torch.randint(-max_pixels, max_pixels + 1, (n, 2), generator=generator)
    theta = torch.eye(2, 3).unsqueeze(0).repeat(n, 1, 1)
    # affine_grid translations are normalized to [-1, 1].
    theta[:, 0, 2] = 2.0 * shifts[:, 1] / batch.shape[-1]
    theta[:, 1, 2] = 2.0 * shifts[:, 0] / batch.shape[-2]
    grid = F.affine_grid(theta, batch.size(), align_corners=False)
    return F.grid_sample(batch, grid, mode="bilinear", padding_mode="zeros", align_corners=False)


original = x_train[:16]
shifted = translate_digits(original)
assert shifted.shape == original.shape
assert shifted.min() >= 0 and shifted.max() <= 1
assert not torch.allclose(shifted, original)

seed_everything(610)
augmented_model = DigitsMLP(widths=(128, 64))
augmented_x = torch.cat([x_train, translate_digits(x_train, seed=611)])
augmented_y = torch.cat([y_train, y_train])
augmented_history = fit_model(
    augmented_model, epochs=25, lr=3e-3, seed=610,
    train_x=augmented_x, train_y=augmented_y,
)
print(augmented_history[-1]["validation"])
```

</details>

更完整的实验应在每个 epoch 重新采样变换，而不是只物化一个副本。这里的紧凑实现突出信息边界：只增强训练数据，绝不能让相关副本落在数据划分的两侧。

### **MixUp、CutMix 与标签平滑** {#mixup-cutmix-label-smoothing}

这些方法以不同方式软化监督信号。

MixUp 选择两个样本并令 $\lambda\sim\mathrm{Beta}(\alpha,\alpha)$：

$$
\widetilde x=\lambda x_i+(1-\lambda)x_j,
\qquad
\widetilde y=\lambda y_i+(1-\lambda)y_j.
$$

目标成为类别概率向量，因此交叉熵是 $-\sum_k\widetilde y_k\log p_k$。MixUp 鼓励模型在训练样本之间近似线性地预测，并降低极端置信度。当插值没有语义意义时，这一假设就不可靠。

CutMix 把一张图像的空间区域替换为另一张图像的区域。目标权重应取**实际保留面积比例**，而不是直接使用采样标量，因为矩形在图像边界被裁剪后面积会变化。CutMix 比像素插值更能保留局部纹理，但 $8\times8$ Digits 也清楚暴露其限制：一个小区域就可能移除大量语义信息。

标签平滑把 one-hot 目标 $q$ 替换为

$$
q'_k=(1-\varepsilon)q_k+\frac{\varepsilon}{K},
$$

其中 $K$ 是类别数。它抑制无限增大的 logit 间隔，但不会创造输入不变性。MixUp/CutMix 已经产生软目标，因此盲目叠加强标签平滑可能导致过度正则化，并让收益来源难以判断。

<details>
<summary><strong>PyTorch：从同一个 Digits batch 构造三种目标</strong></summary>

```python
seed_everything(611)
xb, yb = next(iter(make_loader(x_train, y_train, batch_size=32, shuffle=True, seed=611)))
permutation = torch.randperm(len(xb))
lam = 0.7

one_hot = F.one_hot(yb, num_classes=10).float()
paired = F.one_hot(yb[permutation], num_classes=10).float()
mixup_x = lam * xb + (1 - lam) * xb[permutation]
mixup_y = lam * one_hot + (1 - lam) * paired

cutmix_x = xb.clone()
top, left, height, width = 2, 2, 4, 4
cutmix_x[:, :, top:top + height, left:left + width] = xb[permutation, :, top:top + height, left:left + width]
replaced_fraction = (height * width) / (xb.shape[-2] * xb.shape[-1])
cutmix_y = (1 - replaced_fraction) * one_hot + replaced_fraction * paired

epsilon = 0.1
smoothed_y = (1 - epsilon) * one_hot + epsilon / 10
logits = DigitsMLP(widths=(64,))(mixup_x)
soft_cross_entropy = -(mixup_y * logits.log_softmax(-1)).sum(-1).mean()

assert mixup_x.shape == cutmix_x.shape == xb.shape
assert torch.allclose(mixup_y.sum(1), torch.ones(len(xb)))
assert torch.allclose(cutmix_y.sum(1), torch.ones(len(xb)))
assert torch.allclose(smoothed_y.sum(1), torch.ones(len(xb)))
assert torch.isfinite(soft_cross_entropy)
print({"mixup loss": soft_cross_entropy.item(), "CutMix replaced fraction": replaced_fraction})
```

</details>

三者的差异是概念性的：普通增强改变输入分布；MixUp 和 CutMix 把输入组合与软目标绑定；标签平滑只改变目标置信度。它们是否有效取决于这些假设是否与任务匹配。

### **早停与模型选择** {#early-stopping-model-selection}

早停把优化时间视为正则化参数。每当预先声明的验证指标改善时保存检查点；连续固定次数没有改善后停止；最后恢复最佳检查点，而不是使用最后一次迭代。在线性问题中，提早停止梯度下降会在低信号方向被完全拟合前抑制它们；深层网络没有完全相同的谱解释，但有限训练仍限制了可达解。

三个细节可以避免常见错误：

1. 必须明确监控指标及其方向。准确率要最大化，损失要最小化。
2. patience 按评估事件计数，而不是天然按 epoch 计数。改变评估频率就改变了规则。
3. 优化器与调度器状态属于可恢复检查点，即使最终推理只需要模型权重。

一旦验证集用于决定停止时间，它就不再是无偏评估集。反复查看验证结果、修改 patience 并重跑随机种子，会逐渐让决策过拟合验证集。因此仍然需要锁定测试集。

<details>
<summary><strong>PyTorch：在固定验证集上停止并恢复最佳状态</strong></summary>

```python
seed_everything(612)
early_model = DigitsMLP(widths=(256, 128), dropout=0.1)
best = {"loss": float("inf"), "state": None, "epoch": None}
patience, stale = 8, 0


def early_stopping_callback(model, record):
    global stale
    validation_loss = record["validation"]["loss"]
    if validation_loss < best["loss"] - 1e-4:
        best.update(loss=validation_loss, state=copy.deepcopy(model.state_dict()), epoch=record["epoch"])
        stale = 0
    else:
        stale += 1
    return stale >= patience


early_history = fit_model(
    early_model, epochs=100, lr=3e-3, weight_decay=1e-3,
    seed=612, callback=early_stopping_callback,
)
last_validation = classification_metrics(early_model, x_val, y_val)
early_model.load_state_dict(best["state"])
restored_validation = classification_metrics(early_model, x_val, y_val)

assert best["state"] is not None
assert restored_validation["loss"] <= last_validation["loss"] + 1e-7
print({"stopped_after": len(early_history), "best_epoch": best["epoch"] + 1,
       "restored_validation": restored_validation})
```

</details>

代码使用验证损失，因为它在仅有 360 个样本的验证集上比准确率更平滑。生产研究还应保存优化器、调度器、随机数状态、预处理配置和数据版本。

### **双降与现代泛化现象** {#double-descent-modern-generalization}

经典偏差-方差图景认为，测试误差会随容量增加呈 U 形。**双降（double descent）**在插值阈值之后增加第二次下降：误差可能先降低，在刚好能够拟合训练数据的最小容量附近升高，然后在进一步过参数化后再次下降。

该现象不是每个神经网络实验都必须出现的定律。曲线形状取决于样本量、标签噪声、优化器、训练时长、正则化、架构，以及“容量”所采用的坐标轴。宽度、参数量、训练 epoch 和数据量对应不同的扫描。在插值附近，有限优化还可能让某个宽度看起来更差，只因为它训练得更慢。

相关观察包括 benign overfitting，即插值模型在特定数据/噪声条件下仍可获得低总体误差；以及 grokking，即训练准确率饱和很久后验证表现才突然改善。这些现象说明训练误差和名义参数数量不是完整的状态变量，但它们并没有消除留出评估的必要性。

![示意风险曲线对比了经典 U 形与超过插值阈值后的双降。](assets/dl06-double-descent.png){fig-align="center" width="74%" fig-alt="测试风险示意图，对比经典偏差方差曲线与插值阈值附近的双降曲线。"}

*图片来源：本地教学重绘，依据 [Belkin et al., Reconciling modern machine-learning practice and the classical bias-variance trade-off](https://doi.org/10.1073/pnas.1903070116) 的容量-风险讨论。*

<details>
<summary><strong>PyTorch：在 Digits 上谨慎执行宽度扫描</strong></summary>

```python
width_sweep = []
for width in (4, 16, 64, 256, 512):
    seed_everything(613)
    model = DigitsMLP(widths=(width,))
    history = fit_model(model, epochs=25, lr=3e-3, seed=613)
    parameters = sum(p.numel() for p in model.parameters())
    width_sweep.append({
        "width": width,
        "parameters": parameters,
        "train_error": 1 - history[-1]["train"]["accuracy"],
        "validation_error": 1 - history[-1]["validation"]["accuracy"],
    })

assert [row["parameters"] for row in width_sweep] == sorted(row["parameters"] for row in width_sweep)
assert all(0 <= row["validation_error"] <= 1 for row in width_sweep)
print(width_sweep)
```

</details>

这个小型扫描是**测量模板**，不是 Digits 存在双降的证据。严谨结论需要重复随机种子，让每个宽度达到可比收敛程度，定位插值阈值，改变标签噪声并报告不确定性。预期曲线没有出现时，那同样是结果，而不是隐藏实验的理由。

### **实验协议与超参数选择** {#experimental-protocol-hyperparameter-selection}

实验协议应在最吸引人的结果出现之前写好。它至少要固定数据版本与划分逻辑、主要指标、预处理拟合边界、模型选择预算、停止规则和测试集访问策略。没有这个契约，每次运行都会暗中增加一次选择噪声的机会。

超参数包括架构深度/宽度、增强强度、优化器、学习率、权重衰减、batch size、随机种子策略和停止 patience。网格搜索透明但规模指数增长；当只有少数维度重要时，随机搜索能把更多试验分配给不同取值；贝叶斯与 bandit 方法可能更节省样本，但自适应过程并不会让验证集变得无限，搜索预算仍属于需要报告的方法。

即使完整嵌套交叉验证成本过高，也应采用嵌套式思路：

- **内部开发循环：**在训练数据上拟合，在验证数据上排序配置；
- **确认循环：**用预先规定的多个随机种子重跑选中配置，必要时在训练加验证数据上重新拟合；
- **外部评估：**只在锁定测试集上评估一次。

对小型数据集，交叉验证能降低划分敏感性。深度学习比较常使用重复分层划分或多个随机种子，因为完整嵌套训练代价很高。无论选择哪种设计，都必须报告哪些因素变化、哪些保持固定。

<details>
<summary><strong>PyTorch：不接触测试标签地选择权重衰减</strong></summary>

```python
search_records = []
for decay in (0.0, 1e-4, 1e-3, 1e-2):
    seed_everything(614)
    candidate = DigitsMLP(widths=(128, 64), dropout=0.1)
    history = fit_model(candidate, epochs=25, lr=3e-3, weight_decay=decay, seed=614)
    search_records.append((history[-1]["validation"]["loss"], decay, copy.deepcopy(candidate.state_dict())))

selected_loss, selected_decay, selected_state = min(search_records, key=lambda row: row[0])
selected_model = DigitsMLP(widths=(128, 64), dropout=0.1)
selected_model.load_state_dict(selected_state)
locked_test_result = classification_metrics(selected_model, x_test, y_test)

assert selected_decay in {0.0, 1e-4, 1e-3, 1e-2}
assert 0.0 <= locked_test_result["accuracy"] <= 1.0
print({"selected_decay": selected_decay, "validation_loss": selected_loss,
       "locked_test": locked_test_result})
```

</details>

只有在选择完成后才产生一个测试结果。若按 `x_test` 上最好的表现选择衰减强度，测试集就会退化为另一个验证集，报告准确率也会产生乐观偏差。

### **消融研究、随机种子与实验跟踪** {#ablation-studies-random-seeds-experiment-tracking}

**消融（ablation）**通过移除或改变一个组件，检验完整系统中的因果问题。“模型优于基线”无法说明收益究竟来自架构、更多参数、更长训练、更强增强还是更大的搜索预算。干净的消融需要固定数据、初始化策略、更新次数、评估代码和所有无关选择。

随机种子会影响初始化、小批量顺序、随机正则化，有时还影响非确定性内核。一个种子只是一次实现，而不是置信区间。配对比较应让每种处理使用相同的种子集合，并分析逐种子差值

$$
d_s=M_{A,s}-M_{B,s}.
$$

均值 $\bar d$ 估计所选种子分布下的处理差异，离散程度显示稳定性。只有三到五个种子时，应避免强分布性结论，并展示每次运行，同时说明超参数是在种子研究前还是之后选定的。

实验跟踪应记录配置、代码版本、数据集身份、划分索引或哈希、软件版本、硬件、随机种子、随时间变化的指标和检查点来源。仪表盘很方便，但只要能重建运行，机器可读的本地记录同样足够。

<details>
<summary><strong>PyTorch：跨随机种子执行配对 Dropout 消融</strong></summary>

```python
paired_runs = []
for seed in (615, 616, 617):
    per_seed = {"seed": seed}
    for label, dropout_rate in (("baseline", 0.0), ("dropout", 0.2)):
        seed_everything(seed)
        model = DigitsMLP(widths=(128, 64), dropout=dropout_rate)
        history = fit_model(model, epochs=25, lr=3e-3, weight_decay=1e-3, seed=seed)
        per_seed[label] = history[-1]["validation"]["accuracy"]
    per_seed["difference"] = per_seed["dropout"] - per_seed["baseline"]
    paired_runs.append(per_seed)

differences = torch.tensor([row["difference"] for row in paired_runs])
summary = {
    "mean_difference": differences.mean().item(),
    "sample_std": differences.std(unbiased=True).item(),
    "all_runs": paired_runs,
}
assert len(paired_runs) == 3
assert torch.isfinite(differences).all()
print(summary)
```

</details>

负的平均差异并不说明 Dropout 普遍有害；它只说明该强度在这个架构、数据集、预算和种子集合下没有帮助。这种更窄的陈述正是受控消融能够支持的结论。

### **数据泄漏与可复现性失败** {#data-leakage-reproducibility-failures}

当预期预测时刻不可获得的信息影响训练或模型选择时，就发生了**数据泄漏（data leakage）**。常见渠道包括在全部行上拟合归一化、使用测试标签选择特征、先增强后划分、让同一患者或文档进入多个划分，以及反复针对公开排行榜调参。

正确的划分单位应当是部署所隐含的独立实体。如果同一用户贡献多条记录，按行随机划分会让身份模式跨越边界。时间相关应用通常需要按时间顺序划分，因为未来统计在过去不可获得。近重复图像和派生裁剪也应留在同一组。

可复现性不仅是设置随机种子。种子无法修复缺失的数据版本、改变的预处理、非确定性硬件内核、未记录的过滤规则或被覆盖的检查点。应区分：

- **repeatability：**相同代码、数据、环境和硬件得到接近结果；
- **reproducibility：**独立实现能够支持所声称的结果；
- **replicability/generalization：**结论能在新样本、新站点或新设置中成立。

<details>
<summary><strong>Python：利用来源 ID 检测增强泄漏</strong></summary>

```python
# A tempting but invalid pipeline augments each sample and then randomly splits rows.
base_ids = torch.arange(len(images))
duplicated_images = torch.cat([images, translate_digits(images, seed=618)])
duplicated_targets = torch.cat([targets, targets])
provenance = torch.cat([base_ids, base_ids])

generator = torch.Generator().manual_seed(618)
row_order = torch.randperm(len(duplicated_images), generator=generator)
cut = int(0.8 * len(row_order))
bad_train_ids = set(provenance[row_order[:cut]].tolist())
bad_test_ids = set(provenance[row_order[cut:]].tolist())
cross_boundary_duplicates = bad_train_ids & bad_test_ids

# The chapter pipeline splits base IDs first and augments only inside training.
good_train_ids = set(train_idx.tolist())
good_test_ids = set(test_idx.tolist())

assert len(cross_boundary_duplicates) > 0
assert good_train_ids.isdisjoint(good_test_ids)
print({"leaked identities in bad split": len(cross_boundary_duplicates),
       "leaked identities in chapter split": len(good_train_ids & good_test_ids)})
```

</details>

测试预处理中从未显式复制标签，但相关图像跨越了边界，使记忆变得有用。来源 ID 和分组断言能够发现仅凭指标检查难以识别的一类泄漏。

### **本章对比与总结** {#chapter-comparison-summary}

泛化不能从低训练损失、流行的正则化方法或一次有利的测试结果中直接推断。它是一项关于观测样本之外行为的主张，需要由合理的信息边界和受控证据支持。

| 概念或工具 | 改变的对象 | 应检查的证据 | 典型失败 |
|---|---|---|---|
| 容量与训练时间 | 可达函数与拟合程度 | 匹配的学习曲线 | 仅凭参数量判断“过拟合” |
| 显式正则化 | 目标、标签或输入 | 隔离变量的验证比较 | 同时组合多个因素而失去归因 |
| 隐式正则化 | 优化路径与参数化 | 解的范数、间隔与稳定性 | 假设存在一种普遍隐式偏置 |
| AdamW 权重衰减 | 参数的乘法式收缩 | 参数组、范数和验证曲线 | 衰减所有参数或盲目复制强度 |
| Dropout | 隐藏单元协同适应 | 训练/评估模式行为与种子变化 | 在训练模式评估或移除过多信号 |
| 随机深度 | 残差路径深度 | 路径存活率与深网稳定性 | 在不适合的非残差结构中使用 |
| 数据增强 | 输入变换分布 | 变换样本与切片指标 | 改变真实标签或在划分前增强 |
| MixUp/CutMix | 输入-目标组合几何 | 软目标损失、校准和鲁棒性 | 不合理的混合或错误面积权重 |
| 标签平滑 | 目标置信度 | 准确率与校准 | 无消融地叠加强软目标方法 |
| 早停 | 优化时长 | 验证轨迹与恢复的检查点 | 使用最后而非最佳状态 |
| 超参数搜索 | 模型选择流程 | 声明的空间、预算与锁定测试集 | 反复按测试结果选择 |
| 消融与随机种子 | 因果归因与变异性 | 配对运行与逐种子差异 | 只展示幸运种子 |
| 泄漏控制 | 信息可用性 | 来源、分组/时间划分和拟合边界 | 在相关实体之间随机按行划分 |

Digits 实验构成了一项连贯研究：先建立不可变数据划分，再一次改变容量、范数控制、随机正则化、增强、目标、停止和选择中的一个因素。这种连续性很重要；如果示例之间连数据集和指标都改变，观察到的差异就不能归因于正在讲解的机制。

实践顺序如下：

1. 定义部署总体、预测时刻、独立划分单位和主要指标。
2. 锁定测试集，并且只在训练数据上拟合所有学习型预处理。
3. 建立可复现基线，同时检查损失和面向任务的学习曲线。
4. 因为某种正则化机制的假设与任务匹配而加入它，而不是因为它流行。
5. 在声明的预算内用验证集选择超参数，再通过固定的多个种子确认。
6. 恢复选中检查点，只在锁定测试集上评估一次。
7. 保存配置、来源、环境与结果，使结论能够被审计。

因此，正则化不是一袋互不相关的技巧，而是关于哪些函数有用的假设，以及能够检验该假设的实验。第 07 章继续使用同一 Digits 问题，但核心问题从**模型如何泛化**转向**卷积架构如何表示空间结构**。